# Visitors vs locals — `CUSTOMER_HOME_CITY`

The only column in the whole share that can tell a visitor from a local. For every EFW shop in the
11 host cities this notebook works out what share of its card customers live **outside the host
metro's states**, then writes two small files the map reads:

- `web/visitors.json` — one row per shop: visitor share, customers counted, top home cities
- `web/visitors_city_month.json` — visitor share per city per month, for the seasonal picture

Run top to bottom. Step 1 prints raw examples of the column so we can see its shape before trusting
the parse; if the printout looks different from a dict of `"City, ST": count`, stop and paste it back.

In [ ]:
# ============================================================
# STEP 0 — mount, load the columns we need
# ============================================================
import os, json, re
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount("/content/drive"); CACHE = "/content/drive/MyDrive/ricehack_cache"
except ImportError:
    CACHE = os.path.expanduser("~/ricehack_cache")
WEB = f"{CACHE}/web"; os.makedirs(WEB, exist_ok=True)

COLS = ["PLACEKEY","MARKET","CITY","REGION","LATITUDE","LONGITUDE","NAICS_CODE",
        "SPEND_DATE_RANGE_START","RAW_NUM_CUSTOMERS","CUSTOMER_HOME_CITY"]
df = pd.read_parquet(f"{CACHE}/spend-patterns-rice.parquet", columns=COLS)
print(f"{len(df):,} shop-months")

EFW_NAICS3 = {"722","445","447","721","711","712","713","562","221","311","312","424","485","481","488"}
df["n3"] = df.NAICS_CODE.astype(str).str[:3]
df = df[df.n3.isin(EFW_NAICS3)].copy()
df["month"] = pd.to_datetime(df.SPEND_DATE_RANGE_START).dt.strftime("%Y-%m")
print(f"{len(df):,} EFW shop-months, {df.PLACEKEY.nunique():,} shops")


In [ ]:
# ============================================================
# STEP 1 — LOOK at the column before parsing it
# ============================================================
sample = df.CUSTOMER_HOME_CITY.dropna().head(5)
for v in sample:
    print(type(v).__name__, "->", str(v)[:300]); print()
print("share of rows with a value:", round(df.CUSTOMER_HOME_CITY.notna().mean(), 3))


In [ ]:
# ============================================================
# STEP 2 — parse to {home_city: customers}, decide local vs visitor
# ============================================================
def as_dict(v):
    # {"key": [...], "value": [...]}  (arrow map/struct shape)
    if isinstance(v, dict) and "key" in v and "value" in v:
        return {str(k): float(x) for k, x in zip(v["key"], v["value"]) if x is not None}
    if isinstance(v, dict):
        out = {}
        for k, x in v.items():
            if isinstance(x, (list, np.ndarray)):          # value is a list: take its first number
                x = next((y for y in x if y is not None), None)
            if x is not None:
                try: out[str(k)] = float(x)
                except (TypeError, ValueError): pass
        return out
    if isinstance(v, (list, np.ndarray)):
        out = {}
        for item in v:
            if isinstance(item, dict):
                k = item.get("key") or item.get("city") or item.get("name")
                x = item.get("value") or item.get("count") or item.get("customers")
                if k is not None and x is not None: out[str(k)] = float(x)
        return out
    if isinstance(v, str):
        try: p = json.loads(v)
        except Exception: return {}
        return as_dict(p)
    return {}

# states that count as "local" for each host metro
LOCAL_STATES = {
    "Atlanta": {"GA"}, "Boston": {"MA","NH","RI"}, "Dallas": {"TX"}, "Houston": {"TX"},
    "Kansas City": {"MO","KS"}, "Los Angeles": {"CA"}, "San Francisco Bay Area": {"CA"},
    "Miami": {"FL"}, "New York/New Jersey": {"NY","NJ","CT"}, "Philadelphia": {"PA","NJ","DE"}, "Seattle": {"WA"},
}
state_re = re.compile(r",\s*([A-Z]{2})\s*$")
def split_home(d, market):
    loc = vis = 0.0
    for k, x in d.items():
        m = state_re.search(k)
        st = m.group(1) if m else None
        if st and st in LOCAL_STATES.get(market, set()): loc += x
        else: vis += x
    return loc, vis

parsed = df.CUSTOMER_HOME_CITY.map(as_dict)
lv = [split_home(d, m) for d, m in zip(parsed, df.MARKET)]
df["home_local"]   = [a for a, b in lv]
df["home_visitor"] = [b for a, b in lv]
df["home_total"]   = df.home_local + df.home_visitor
df["top_home"] = parsed.map(lambda d: ", ".join(f"{k} {int(x)}" for k, x in sorted(d.items(), key=lambda kv: -kv[1])[:3]))
print("rows with any home-city customers:", f"{(df.home_total>0).mean():.1%}")
print("overall visitor share:", f"{df.home_visitor.sum()/df.home_total.sum():.1%}")
print()
print("visitor share by host city (all months):")
g = df.groupby("MARKET")[["home_local","home_visitor"]].sum()
print((g.home_visitor/(g.home_local+g.home_visitor)).round(3).sort_values(ascending=False).to_string())


In [ ]:
# ============================================================
# STEP 3 — per shop (2022-2024) and per city-month, then write for the map
# ============================================================
recent = df[df.month >= "2022-01"]
shop = recent.groupby("PLACEKEY").agg(local=("home_local","sum"), visitor=("home_visitor","sum"),
                                       n_months=("month","nunique"))
shop["share"] = shop.visitor / (shop.local + shop.visitor)
shop = shop[(shop.local + shop.visitor) >= 20]          # need a few dozen customers to say anything
top = recent.sort_values("home_total", ascending=False).drop_duplicates("PLACEKEY").set_index("PLACEKEY").top_home
shop["top"] = top.reindex(shop.index).fillna("")
out = [{"k": k, "v": round(r.share, 3), "n": int(r.local + r.visitor), "t": r.top} for k, r in shop.iterrows()]
json.dump(out, open(f"{WEB}/visitors.json", "w"), separators=(",", ":"))
print(f"visitors.json: {len(out):,} shops, {os.path.getsize(f'{WEB}/visitors.json')/1e6:.1f} MB")

cm = df.groupby(["MARKET","month"])[["home_local","home_visitor"]].sum().reset_index()
cm["share"] = cm.home_visitor / (cm.home_local + cm.home_visitor)
json.dump([{"m": r.MARKET, "d": r.month, "v": round(r.share, 4)} for r in cm.itertuples()],
          open(f"{WEB}/visitors_city_month.json", "w"), separators=(",", ":"))
print("visitors_city_month.json written")

print("\nJune vs the rest of the year — do visitors rise in summer? (share of customers from outside the metro's states)")
cm["mon"] = cm.month.str[5:]
piv = cm.pivot_table(index="MARKET", columns="mon", values="share").round(3)
print(piv[["01","04","06","07","10"]].to_string())
print("\ndownload web/visitors.json and web/visitors_city_month.json into app/data/")
